In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
fake_df = pd.read_csv('Fake.csv')
true_df = pd.read_csv('True.csv')
fake_df.info()
true_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


In [3]:
fake_df['label']=0
true_df['label']=1

In [4]:
fake_df.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [5]:
true_df.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [6]:
df = pd.concat([fake_df,true_df],ignore_index=True)
df.drop(columns = ['title','subject','date'],inplace = True)
print(df.head())
print(df.shape)

                                                text  label
0  Donald Trump just couldn t wish all Americans ...      0
1  House Intelligence Committee Chairman Devin Nu...      0
2  On Friday, it was revealed that former Milwauk...      0
3  On Christmas day, Donald Trump announced that ...      0
4  Pope Francis used his annual Christmas Day mes...      0
(44898, 2)


In [7]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [8]:
df

,text,label
0,"21st Century Wire says Ben Stein, reputable pr...",0
1,WASHINGTON (Reuters) - U.S. President Donald T...,1
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,1
3,"On Monday, Donald Trump once again embarrassed...",0
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",1
...,...,...
44893,,0
44894,LONDON/TOKYO (Reuters) - British Prime Ministe...,1
44895,BERLIN (Reuters) - Chancellor Angela Merkel sa...,1
44896,Jesus f*cking Christ our President* is a moron...,0


In [9]:
df.duplicated().sum()

np.int64(6251)

In [10]:
df = df.drop_duplicates()
df.shape

(38647, 2)

In [34]:
df

,text,label
0,"21st Century Wire says Ben Stein, reputable pr...",0
1,WASHINGTON (Reuters) - U.S. President Donald T...,1
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,1
3,"On Monday, Donald Trump once again embarrassed...",0
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",1
...,...,...
44890,NAIROBI (Reuters) - Burundi s main opposition ...,1
44892,Miss Universe 1996 Alicia Machado is now an Am...,0
44894,LONDON/TOKYO (Reuters) - British Prime Ministe...,1
44895,BERLIN (Reuters) - Chancellor Angela Merkel sa...,1


In [11]:
y=df["label"]

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

sequences = tokenizer.texts_to_sequences(df["text"])
print(sequences)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [17]:
print("Number of documents:", len(sequences))

Number of documents: 38647


In [18]:
padded_sequences = pad_sequences(sequences, maxlen=10, padding='post')
print(padded_sequences)
print(padded_sequences.shape)

[[4735   79  215 ...  400 1749  873]
 [  12 1129   17 ...  270  621    1]
 [ 251    2  157 ...    5    1  797]
 ...
 [  14 2056 3412 ... 1550    1    1]
 [   4  411    9 ...    5 1231 2789]
 [  76   26  168 ...  484  441 2980]]
(38647, 10)


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()

# Embedding layer CREATES the 3D structure
model.add(
    Embedding(
        input_dim=5000,
        output_dim=128,
        input_length=10
    )
)
# LSTM now receives 3D input
model.add(LSTM(64))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.fit(padded_sequences, y, epochs=7)

Epoch 1/7
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 28s 19ms/step - accuracy: 0.9350 - loss: 0.1682
Epoch 2/7
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 23s 19ms/step - accuracy: 0.9566 - loss: 0.1098
Epoch 3/7
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 41s 18ms/step - accuracy: 0.9680 - loss: 0.0822
Epoch 4/7
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 30s 25ms/step - accuracy: 0.9757 - loss: 0.0597
Epoch 5/7
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 26s 21ms/step - accuracy: 0.9824 - loss: 0.0429
Epoch 6/7
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 37s 18ms/step - accuracy: 0.9885 - loss: 0.0293
Epoch 7/7
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.9923 - loss: 0.0199


In [25]:
prediction = model.predict(padded_sequences)

1208/1208 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step


In [26]:
prediction

array([[3.7246355e-06],
       [9.9744540e-01],
       [1.0000000e+00],
       ...,
       [9.9999493e-01],
       [9.9999970e-01],
       [1.6436500e-08]], shape=(38647, 1), dtype=float32)

In [35]:
label = "Real" if prediction[4][0] >= 0.5 else "Fake"
print(label)

Real
